# T168 — Ranking Olist Data in MySQL

D165, D166, and D167 introduced ranking functions with simple examples. This notebook applies them to `olist_order_items`. It uses seller, product, price, freight, and shipping-month examples without joining another table.

## 1. Connect and define the query helper

In [ ]:
import os
import mysql.connector

connection = mysql.connector.connect(
    host=os.environ.get('MYSQL_HOSTNAME', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USERNAME', 'root'),
    password=os.environ.get('MYSQL_PASSWORD', 'root'),
    database=os.environ.get('MYSQL_DATABASE', 'olist_import_lab'),
)
print('Connected:', connection.is_connected())

In [ ]:
def execute_sql(sql, params=None, max_rows=25):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        columns = [column[0] for column in cursor.description]
        rows = cursor.fetchmany(max_rows + 1)
        has_more = len(rows) > max_rows
        rows = rows[:max_rows]
        values = [[str(value) for value in row] for row in rows]
        widths = [len(column) for column in columns]
        for row in values:
            widths = [max(width, len(value)) for width, value in zip(widths, row)]
        print(' | '.join(c.ljust(w) for c, w in zip(columns, widths)))
        print('-+-'.join('-' * w for w in widths))
        for row in values:
            print(' | '.join(v.ljust(w) for v, w in zip(row, widths)))
        if has_more:
            print(f'... showing the first {max_rows} rows')
        return rows
    finally:
        cursor.close()

## 2. Rank sellers by item value

The CTE first creates one row per seller. The outer query ranks those seller totals. Ranking raw item rows would answer a different question.

In [ ]:
execute_sql("""
WITH seller_totals AS (
    SELECT seller_id, COUNT(*) AS items_sold,
           COUNT(DISTINCT order_id) AS orders_served,
           SUM(price) AS item_value
    FROM olist_order_items
    GROUP BY seller_id
)
SELECT seller_id, items_sold, orders_served, ROUND(item_value, 2) AS item_value,
       RANK() OVER (ORDER BY item_value DESC) AS seller_rank
FROM seller_totals
ORDER BY seller_rank, seller_id
LIMIT 15
""")

## 3. Compare the three basic ranking functions

The total is rounded before ranking so equal displayed totals are treated as ties. `ROW_NUMBER` remains unique, `RANK` leaves gaps after ties, and `DENSE_RANK` does not leave gaps. `seller_id` is used only to break ties for `ROW_NUMBER`.

In [ ]:
execute_sql("""
WITH seller_totals AS (
    SELECT seller_id, ROUND(SUM(price), 0) AS rounded_value
    FROM olist_order_items
    GROUP BY seller_id
)
SELECT seller_id, rounded_value,
       ROW_NUMBER() OVER (ORDER BY rounded_value DESC, seller_id) AS row_number_value,
       RANK()       OVER (ORDER BY rounded_value DESC) AS rank_value,
       DENSE_RANK() OVER (ORDER BY rounded_value DESC) AS dense_rank_value
FROM seller_totals
ORDER BY rounded_value DESC, seller_id
LIMIT 25
""")

## 4. Create seller value bands with `NTILE`

`NTILE(4)` places sellers into four similarly sized groups. Bucket 1 contains the sellers with the highest item values. These are row-count buckets, not four equal money ranges.

In [ ]:
execute_sql("""
WITH seller_totals AS (
    SELECT seller_id, SUM(price) AS item_value
    FROM olist_order_items GROUP BY seller_id
), bucketed AS (
    SELECT seller_id, item_value,
           NTILE(4) OVER (ORDER BY item_value DESC) AS value_bucket
    FROM seller_totals
)
SELECT value_bucket, COUNT(*) AS sellers,
       ROUND(MIN(item_value), 2) AS minimum_value,
       ROUND(MAX(item_value), 2) AS maximum_value,
       ROUND(AVG(item_value), 2) AS average_value
FROM bucketed
GROUP BY value_bucket
ORDER BY value_bucket
""")

## 5. Relative seller position

`PERCENT_RANK` shows relative rank from 0 to 1. `CUME_DIST` shows the cumulative share of sellers through the current value. The query displays percentages for the highest-value sellers.

In [ ]:
execute_sql("""
WITH seller_totals AS (
    SELECT seller_id, SUM(price) AS item_value
    FROM olist_order_items GROUP BY seller_id
), positioned AS (
    SELECT seller_id, item_value,
           PERCENT_RANK() OVER (ORDER BY item_value DESC) AS percent_rank_value,
           CUME_DIST() OVER (ORDER BY item_value DESC) AS cumulative_value
    FROM seller_totals
)
SELECT seller_id, ROUND(item_value, 2) AS item_value,
       ROUND(percent_rank_value * 100, 3) AS percent_rank_pct,
       ROUND(cumulative_value * 100, 3) AS cumulative_percent
FROM positioned
ORDER BY item_value DESC
LIMIT 15
""")

## 6. Rank products inside each seller

`PARTITION BY seller_id` restarts the ranking for every seller. The CTE first makes one row for each seller-product pair. This query shows the three highest-value products for sellers having at least 100 item rows.

In [ ]:
execute_sql("""
WITH active_sellers AS (
    SELECT seller_id FROM olist_order_items
    GROUP BY seller_id HAVING COUNT(*) >= 100
), seller_products AS (
    SELECT i.seller_id, i.product_id, COUNT(*) AS items_sold, SUM(i.price) AS item_value
    FROM olist_order_items i
    JOIN active_sellers a ON a.seller_id = i.seller_id
    GROUP BY i.seller_id, i.product_id
), ranked_products AS (
    SELECT seller_id, product_id, items_sold, item_value,
           ROW_NUMBER() OVER (
               PARTITION BY seller_id
               ORDER BY item_value DESC, product_id
           ) AS product_row_number
    FROM seller_products
)
SELECT seller_id, product_id, items_sold, ROUND(item_value, 2) AS item_value,
       product_row_number
FROM ranked_products
WHERE product_row_number <= 3
ORDER BY seller_id, product_row_number
LIMIT 24
""")

## 7. Rank the most expensive item in each month

This example partitions item rows by shipping month. `ROW_NUMBER` selects exactly one item per month. The IDs provide a stable tie-break when two items have the same price.

In [ ]:
execute_sql("""
WITH monthly_items AS (
    SELECT DATE_FORMAT(shipping_limit_date, '%Y-%m') AS shipping_month,
           order_id, order_item_id, product_id, price,
           ROW_NUMBER() OVER (
               PARTITION BY DATE_FORMAT(shipping_limit_date, '%Y-%m')
               ORDER BY price DESC, order_id, order_item_id
           ) AS month_row_number
    FROM olist_order_items
)
SELECT shipping_month, order_id, order_item_id, product_id, price
FROM monthly_items
WHERE month_row_number = 1
ORDER BY shipping_month
""")

## 8. Top three items with ties

Use `DENSE_RANK` when all rows tied at the third distinct price should be included. The result can contain more than three rows because several items may share a price.

In [ ]:
execute_sql("""
WITH ranked_items AS (
    SELECT order_id, order_item_id, product_id, price,
           DENSE_RANK() OVER (ORDER BY price DESC) AS price_rank
    FROM olist_order_items
)
SELECT order_id, order_item_id, product_id, price, price_rank
FROM ranked_items
WHERE price_rank <= 3
ORDER BY price_rank, order_id, order_item_id
""")

## 9. Practical rules

- Decide the row grain before ranking. Rank seller totals for a seller leaderboard, not raw item rows.
- Use `PARTITION BY` when each business group needs its own ranking.
- Add a stable tie-break to `ROW_NUMBER` when exactly N rows are required.
- Use `RANK` or `DENSE_RANK` when tied business values should share a position.
- Calculate the ranking in a CTE before filtering for top-N results.
- `NTILE` makes similarly sized row groups; it does not create equal value ranges.

In [ ]:
if connection.is_connected():
    connection.close()
print('MySQL connection closed.')